<a href="https://colab.research.google.com/github/kumarpal1107/pythonbasics/blob/main/Imagesegmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1
The TensorFlow Object Detection API (TFOD2) is an open-source framework built on TensorFlow, simplifying the creation, training, and deployment of object detection models with pre-trained models (Model Zoo) and tools, featuring key components like Model Zoo, Pipelines, Configuration Files, Training Scripts, and Export Tools, enabling fast localization and classification in images/videos.

In [ ]:
#2
Semantic segmentation classifies every pixel into a category (e.g., all cars are "car"), providing scene understanding, while instance segmentation separates and labels each individual object of a class (e.g., car_1, car_2), enabling object counting and differentiation, crucial for tasks needing object-level precision like autonomous driving or robotics where distinguishing between multiple vehicles is key.

In [ ]:
#3
Mask R-CNN is an extension of Faster R-CNN for instance segmentation, adding a parallel mask prediction branch to output pixel-level masks for each detected object, alongside bounding boxes and class labels, using a shared backbone (like ResNet) for feature extraction, a Region Proposal Network (RPN), and crucial components like Feature Pyramid Networks (FPN) for scale invariance and RoIAlign for precise feature alignment, overcoming pooling inaccuracies for better segmentation quality.

In [ ]:
#4In image segmentation, masks are pixel-level maps identifying regions of interest (ROIs) or objects, crucial for training models to understand what and where things are, acting as ground truth during training (comparing predictions to actual pixel labels) and generating precise object outlines during inference for applications like autonomous driving or medical analysis.

During training, models learn to map input images to these target masks, minimizing errors; during inference, the trained model outputs its own predicted masks, often refined to match the input image's size, to segment new images.

In [ ]:
#5
Training a custom image segmentation model using the TensorFlow Object Detection (TFOD) API 2 involves several key steps:

The process begins with data collection and annotation, where you gather images and use a tool like LabelMe to create pixel-level segmentation masks for your custom objects.

Next, we prepare the dataset by dividing the annotated images into training and testing sets and converting them into the TFRecord format, which is required by the TFOD API. We also need to create a label map file (label_map.pbtxt) that maps class names to numeric IDs.

Then, we configure the training pipeline by selecting a pre-trained model (like Mask R-CNN) from the TensorFlow 2 Detection Model Zoo and modifying its configuration file to point to your data, specify the number of classes, and adjust hyperparameters.

The next step is to train the model using the configured pipeline and data, monitoring its progress and metrics (like loss) with TensorBoard.

After training is complete, we export the trained model's weights to create an inference graph.

Finally, we can test the model by running inference on new images to evaluate its performance and visualize the segmentation results.

In [ ]:
#6
# Git Clone Github Project
!git clone https://github.com/tensorflow/models.git

# Lets go Inside the Models/research folder
%cd models/research

!pwd

# Protos conversion to python
!protoc object_detection/protos/*.proto --python_out=.

# Getting the Setup File
!cp object_detection/packages/tf2/setup.py .


# Installing the Setup
!pip install .


# Installing other dependencies
!pip install tf-models-official


import os
# Verification: Print available model configs
print("\n--- Verifying Installation: Available Model Configs ---")
config_dir = 'object_detection/configs/tf2'
if os.path.exists(config_dir):
    configs = [f for f in os.listdir(config_dir) if f.endswith('.config')]
    print(f"Found {len(configs)} model configurations:")
    for cfg in sorted(configs):
        print(f" - {cfg}")
else:
    print("Error: Model configuration directory not found.")

In [ ]:
#7
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt


import requests

url = "https://images.pexels.com/photos/13872248/pexels-photo-13872248.jpeg"
image_path = "/content/sample_image.jpg"

response = requests.get(url)
with open(image_path, "wb") as f:
    f.write(response.content)

print("Image downloaded successfully")


import matplotlib.pyplot as plt
import cv2

img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6,6))
plt.imshow(img)
plt.axis("off")
plt.show()


from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

url = "https://images.pexels.com/photos/13872248/pexels-photo-13872248.jpeg"
img = Image.open(BytesIO(requests.get(url).content))

plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
#9


import os
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 1. Path to your COCO ground truth and detection results
# You can generate 'detections.json' by running the model on the test set
ann_file = 'path/to/kangaroo_coco_annotations.json'
det_file = 'path/to/detections.json'

def plot_precision_recall_curve(ann_file, det_file):
    # Load ground truth and detections
    coco_gt = COCO(ann_file)
    coco_dt = coco_gt.loadRes(det_file)

    # Initialize COCOeval object (iouType='segm' for Mask R-CNN, 'bbox' for boxes)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='segm')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    # Extract Precision-Recall data
    # precision has shape [T, R, K, A, M]
    # T: iou thresholds [0.5:0.05:0.95] (index 0 is 0.5)
    # R: recall thresholds [0:0.01:1]
    # K: categories
    # A: area ranges
    # M: max detections
    precision = coco_eval.eval['precision'][0, :, 0, 0, 2] # IoU=0.5, All recall, Category 0
    recall = np.linspace(0, 1, 101)

    # Plot the curve
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color='blue', lw=2, label='Mask R-CNN (IoU=0.5)')
    plt.fill_between(recall, precision, alpha=0.2, color='blue')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve (Mask R-CNN)')
    plt.legend(loc="lower left")
    plt.grid(True)
    plt.show()

# Run the plotting function
# plot_precision_recall_curve(ann_file, det_file)

In [ ]:
#10
To improve illegal parking detection around sidewalk boundaries, I would transition from simple bounding boxes to Instance Segmentation using models like Mask R-CNN or YOLOv8/v11, which provide pixel-level masks to accurately delineate cars from sidewalks.

To handle complex street scenes, I would employ a multi-task learning strategy that simultaneously segments cars, sidewalks, and curblines, using DeepLabV3+ for high-resolution segmentation.

For training, I would utilize semantic segmentation data with polygonal annotations created in Labelbox or V7 Labs, and apply augmentation techniques like random cropping, rotation, and lighting changes to improve robustness against occlusions and varied weather.

Finally, I would incorporate a post-processing step to calculate the exact percentage of the car mask overlapping with the labeled sidewalk mask, refining the detection of illegal parking.